|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>The async engine<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: build the engine loop<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import asyncio
import time

STEP_S = 0.010
print('ready')

Build the engine loop.

One coroutine steps the model forever. Requests arrive on a queue, and tokens
leave on one queue for each request. Nothing blocks in either direction.

This is stage 15, with `asyncio.sleep` in the place of the model. Make the
concurrency correct before you add the GPU.

# Exercise 1: submit, step, stream

Three pieces: a way in, a loop, and a way out. The loop is the only thing
that touches the model.

In [ ]:
class Engine:
  def __init__(self, max_running=8):
    self.waiting     = asyncio.Queue()
    self.running     = {}          # rid -> [remaining, out_queue]
    self.max_running = max_running
    self.stop        = False

  async def submit(self, rid, n_tokens):
    """HTTP calls this. It must return immediately."""
    q = asyncio.Queue()
    
    return q

  def _admit(self):
    # move waiting requests into running while there is room
    

  async def loop(self):
    """The only thing that touches the model. One step serves everybody."""
    while not self.stop:
      self._admit()
      if not self.running:
        await asyncio.sleep(STEP_S); continue
      await asyncio.sleep(STEP_S)              # the step
      # give every running request its token, and close the ones that
      # just finished
      

  def cancel(self, rid):
    """A client hung up. Free the slot NOW, not when it would have ended."""
    

eng = Engine(max_running=4)
task = asyncio.create_task(eng.loop())

async def client(rid, n):
  t0 = time.perf_counter(); q = await eng.submit(rid, n); first = None
  while True:
    tok = await q.get()
    if tok is None: break
    if first is None: first = time.perf_counter() - t0
  return first, time.perf_counter() - t0

res = await asyncio.gather(*[client(i, 10) for i in range(8)])
eng.stop = True; await asyncio.sleep(0.05); task.cancel()
print(f'{len(res)} clients done')
print(f'TTFT  min {min(a for a,_ in res):.3f}s  max {max(a for a,_ in res):.3f}s')

# Exercise 2: somebody closes the tab

Watch the number of occupied slots against time. Two clients hang up after
five tokens. Their slots must come back at once, and not at the time when the
request would have finished.

In [ ]:
eng = Engine(max_running=4)
task = asyncio.create_task(eng.loop())
occupied = []

async def watcher():
  for _ in range(40):
    occupied.append(len(eng.running))
    await asyncio.sleep(STEP_S)

async def quitter(rid, n, after):
  q = await eng.submit(rid, n)
  for k in range(after):
    if await q.get() is None: return
  # the tab closed. What has to happen here?
  

async def stayer(rid, n):
  q = await eng.submit(rid, n)
  while await q.get() is not None: pass

await asyncio.gather(watcher(),
                     *[quitter(i, 30, 5) for i in (0,1)],
                     *[stayer(i, 30) for i in (2,3)])
eng.stop = True; await asyncio.sleep(0.05); task.cancel()

print('slots in use over time:', occupied[:20])
print(f'\npeak {max(occupied)}')

# Exercise 3: against the obvious design

Now the version where the model runs inside the event loop. Measure time to
first token, not throughput.

In [ ]:
async def measure(design, n_clients=8, tokens=15):
  T0 = time.perf_counter(); lat = []
  if design == 'blocking':
    # the model runs inside the event loop: time.sleep, not asyncio.sleep
    async def c(i):
      f = None
      for _ in range(tokens):
        
      lat.append(f)
    await asyncio.gather(*[c(i) for i in range(n_clients)])
  else:
    e = Engine(max_running=n_clients)
    t = asyncio.create_task(e.loop())
    async def c(i):
      
    await asyncio.gather(*[c(i) for i in range(n_clients)])
    e.stop = True; await asyncio.sleep(0.05); t.cancel()
  return sorted(lat), time.perf_counter()-T0

for design in ('blocking', 'engine'):
  lat, wall = await measure(design)
  print(f'{design:>9}: wall {wall:5.2f}s  TTFT p50 {lat[len(lat)//2]:.3f}s  p99 {lat[-1]:.3f}s')

### Before you open the solution

1. `submit` returns a queue rather than the tokens. Why can it not just
   await the result and return it?
2. Delete the body of `cancel` and rerun Exercise 2. What does the
   occupancy trace look like, and what is the server holding?
3. The blocking design in Exercise 3 uses `asyncio.gather`, which looks
   concurrent. Why is it not?
4. The engine sends `None` to end a stream. What would break if it simply
   stopped sending?